# Tool Use & Function Calling

This file teaches you 7 patterns for giving AI models access to tools.

**Function calling** means the model can ask to run a function (like a calculator or API call). The model does not run the function itself -- it tells you which function to call and what arguments to use. You run it and send the result back.

By the end, you will know how to use single tools, parallel tools, chained tools, nested tools, dynamically created tools, tool retrieval, and MCP.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
from openai import OpenAI
import json
import time

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

**What happened:** We imported the OpenAI library and set up our client.

## Shared Tools

We define several tools that we reuse across all examples in this notebook.

In [4]:
def calculator(expression):
    result = eval(expression)
    return str(result)



def get_weather(city):
    weather_data = {
        "new york": {"temp": 72, "condition": "sunny", "humidity": 45},
        "london": {"temp": 59, "condition": "rainy", "humidity": 80},
        "tokyo": {"temp": 68, "condition": "cloudy", "humidity": 60},
        "paris": {"temp": 65, "condition": "partly cloudy", "humidity": 55},
        "sydney": {"temp": 78, "condition": "sunny", "humidity": 40},
    }
    
    city_lower = city.lower()
    
    if city_lower in weather_data:
        
        data = weather_data[city_lower]
        result = city + ": " + str(data["temp"]) + "F, " + data["condition"] + ", " + str(data["humidity"]) + "% humidity"
        return result
    
    return "Weather data not available for " + city



def get_time(timezone):
    
    times = {
        "EST": "2:00 PM",
        "GMT": "7:00 PM",
        "JST": "4:00 AM",
        "CET": "8:00 PM",
        "AEST": "5:00 AM",
    }
    
    if timezone in times:
        return timezone + ": " + times[timezone]
    
    return "Time not available for " + timezone



def translate_text(text, target_language):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Translate the following text to " + target_language + ". Reply with ONLY the translation."
            },
            {"role": "user", "content": text}
        ],
        max_tokens=200
    )
    translation = response.choices[0].message.content
    return translation



def search_products(query):
    
    products = [
        {"name": "Wireless Mouse", "price": 29.99, "category": "electronics"},
        {"name": "Mechanical Keyboard", "price": 79.99, "category": "electronics"},
        {"name": "USB-C Hub", "price": 34.99, "category": "electronics"},
        {"name": "Desk Lamp", "price": 24.99, "category": "home"},
        {"name": "Standing Desk Mat", "price": 39.99, "category": "home"},
    ]
    
    
    results = []
    query_lower = query.lower()
    
    for product in products:
        name_lower = product["name"].lower()
        category_lower = product["category"].lower()
        if query_lower in name_lower or query_lower in category_lower:
            results.append(product)
    
    if len(results) == 0:
        return "No products found for: " + query
    
    text = ""
    for product in results:
        text = text + product["name"] + " - $" + str(product["price"]) + "\n"
    return text.strip()



def send_email(to, subject, body):
    print("  [Simulated] Sending email to:", to)
    print("  Subject:", subject)
    print("  Body:", body[:100])
    return "Email sent successfully to " + to



print("6 tools defined:")
print("  calculator, get_weather, get_time,")
print("  translate_text, search_products, send_email")

6 tools defined:
  calculator, get_weather, get_time,
  translate_text, search_products, send_email


### Tool definitions for OpenAI

In [5]:
all_tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '25 * 4'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name like 'New York'"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Get the current time in a timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "Timezone like 'EST', 'GMT', 'JST'"}
                },
                "required": ["timezone"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "translate_text",
            "description": "Translate text to another language.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to translate"},
                    "target_language": {"type": "string", "description": "Language to translate to, like 'Spanish'"}
                },
                "required": ["text", "target_language"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search for products by name or category.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query like 'keyboard' or 'electronics'"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to someone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Email address"},
                    "subject": {"type": "string", "description": "Email subject"},
                    "body": {"type": "string", "description": "Email body text"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

print("Registered", len(all_tools), "tools with OpenAI format")

Registered 6 tools with OpenAI format


**What happened:** We described all 6 tools in OpenAI's function calling format. Each tool has a name, description, and typed parameters. The model reads these to decide which tool to call.

### Helper: run any tool by name

In [6]:
def run_tool(function_name, arguments):
    if function_name == "calculator":
        return calculator(arguments["expression"])
    elif function_name == "get_weather":
        return get_weather(arguments["city"])
    elif function_name == "get_time":
        return get_time(arguments["timezone"])
    elif function_name == "translate_text":
        return translate_text(arguments["text"], arguments["target_language"])
    elif function_name == "search_products":
        return search_products(arguments["query"])
    elif function_name == "send_email":
        return send_email(arguments["to"], arguments["subject"], arguments["body"])
    else:
        return "Unknown tool: " + function_name

print("run_tool helper is ready")

run_tool helper is ready


**What happened:** We created a helper that takes a tool name and arguments and runs the correct tool function.

## 1. Single Tool Selection

**What it does:** The model picks **one tool** per turn, calls it, and uses the result to answer.

**When to use it:** For simple questions that need one piece of information -- one calculation, one lookup, or one search.

**Steps:**
1. Send the question to the model with all available tools.
2. The model picks one tool (or none).
3. Run the tool and send the result back.
4. The model writes the final answer.

In [7]:
def single_tool_call(question):
    print("Question:", question)
    print()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": question}
        ],
        tools=all_tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls is None:
        print("No tool needed. Direct answer:")
        print(message.content)
        return message.content

    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print("Tool selected:", function_name)
    print("Arguments:", arguments)

    tool_result = run_tool(function_name, arguments)
    print("Tool result:", tool_result)
    print()

    # Send result back to model for final answer
    messages = [
        {"role": "user", "content": question},
        message,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        }
    ]

    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    final_answer = final_response.choices[0].message.content
    print("Answer:", final_answer)
    return final_answer

**What happened:** We built a function that sends the question with tools, runs whichever tool the model picks, sends the result back, and gets the final answer.

In [8]:
single_tool_call("What is 1547 * 23 + 892?")

Question: What is 1547 * 23 + 892?

Tool selected: calculator
Arguments: {'expression': '1547 * 23 + 892'}
Tool result: 36473

Answer: The result of \( 1547 \times 23 + 892 \) is 36,473.


'The result of \\( 1547 \\times 23 + 892 \\) is 36,473.'

**What happened:** The model chose `calculator`, sent it the expression, and wrote the final answer using the result.

In [9]:
single_tool_call("What is the weather like in Tokyo?")

Question: What is the weather like in Tokyo?

Tool selected: get_weather
Arguments: {'city': 'Tokyo'}
Tool result: Tokyo: 68F, cloudy, 60% humidity

Answer: The weather in Tokyo is currently 68°F (about 20°C) with cloudy skies and a humidity level of 60%.


'The weather in Tokyo is currently 68°F (about 20°C) with cloudy skies and a humidity level of 60%.'

**What happened:** The model chose `get_weather` instead of calculator. It picked the right tool based on the question.

## 2. Parallel Tool Calling

**What it does:** The model calls **multiple tools at once** in one turn, instead of calling them one at a time.

**When to use it:** When the question needs information from multiple independent sources, like "weather in Tokyo AND London."

**Steps:**
1. Send the question with all available tools.
2. The model requests multiple tool calls at once.
3. Run all tools and collect results.
4. Send all results back for a combined answer.

In [11]:
def parallel_tool_call(question):
    print("Question:", question)
    print()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": question}
        ],
        tools=all_tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls is None:
        print("No tools needed.")
        print(message.content)
        return message.content

    print("Model requested", len(message.tool_calls), "tool calls in parallel:")
    print()

    # Run all tool calls
    messages = [
        {"role": "user", "content": question},
        message
    ]

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print("  Tool:", function_name, "| Args:", arguments)

        tool_result = run_tool(function_name, arguments)
        print("  Result:", tool_result)
        print()

        tool_message = {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        }
        messages.append(tool_message)

    # Get final answer
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    final_answer = final_response.choices[0].message.content
    print("Answer:", final_answer)
    return final_answer

**What happened:** We built a function that handles multiple tool calls from one response. The key difference from single tool selection: we loop through ALL tool calls in `message.tool_calls`.

In [12]:
parallel_tool_call("What is the weather in Tokyo and London, and what time is it in EST and GMT?")

Question: What is the weather in Tokyo and London, and what time is it in EST and GMT?

Model requested 4 tool calls in parallel:

  Tool: get_weather | Args: {'city': 'Tokyo'}
  Result: Tokyo: 68F, cloudy, 60% humidity

  Tool: get_weather | Args: {'city': 'London'}
  Result: London: 59F, rainy, 80% humidity

  Tool: get_time | Args: {'timezone': 'EST'}
  Result: EST: 2:00 PM

  Tool: get_time | Args: {'timezone': 'GMT'}
  Result: GMT: 7:00 PM

Answer: The current weather is as follows:
- **Tokyo**: 68°F, cloudy, with 60% humidity.
- **London**: 59°F, rainy, with 80% humidity.

As for the time:
- **Eastern Standard Time (EST)**: 2:00 PM
- **Greenwich Mean Time (GMT)**: 7:00 PM


'The current weather is as follows:\n- **Tokyo**: 68°F, cloudy, with 60% humidity.\n- **London**: 59°F, rainy, with 80% humidity.\n\nAs for the time:\n- **Eastern Standard Time (EST)**: 2:00 PM\n- **Greenwich Mean Time (GMT)**: 7:00 PM'

**What happened:** The model requested 4 tool calls at once (2 weather, 2 time). All were independent, so it asked for them in parallel. We ran all 4 and the model combined the results.

## 3. Sequential Tool Chaining

**What it does:** The output of one tool becomes the input for the next. The model calls tool A, gets the result, then uses it to decide what to pass to tool B.

**When to use it:** When steps depend on each other, like "search for products and email me the results."

**Steps:**
1. Send the question and tools to the model.
2. The model calls a tool and gets a result.
3. The model uses that result to decide the next tool call.
4. Repeat until the model gives a final answer.

In [13]:
def sequential_chain(question):
    print("Question:", question)
    print()

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Use tools step by step when needed."
        },
        {"role": "user", "content": question}
    ]

    
    for step in range(1, 6):
        print("--- Step", step, "---")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=all_tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        
        if message.tool_calls is None:
            print("Final answer:")
            print(message.content[:400])
            return message.content

        
        messages.append(message)

        
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print("  Tool:", function_name, "| Args:", arguments)

            tool_result = run_tool(function_name, arguments)
            print("  Result:", tool_result[:200])

            tool_message = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            }
            messages.append(tool_message)

        print()

    return None

**What happened:** We built a loop that keeps calling the model until it stops requesting tools. Each step, the model sees all previous tool results, creating a chain where Tool A's result feeds into Tool B's input.

In [14]:
sequential_chain(
    "Search for electronics products, calculate the total cost of all products found, "
    "and then translate the product list to Spanish."
)

Question: Search for electronics products, calculate the total cost of all products found, and then translate the product list to Spanish.

--- Step 1 ---
  Tool: search_products | Args: {'query': 'electronics'}
  Result: Wireless Mouse - $29.99
Mechanical Keyboard - $79.99
USB-C Hub - $34.99

--- Step 2 ---
  Tool: calculator | Args: {'expression': '29.99 + 79.99 + 34.99'}
  Result: 144.97

--- Step 3 ---
  Tool: translate_text | Args: {'text': 'Wireless Mouse - $29.99, Mechanical Keyboard - $79.99, USB-C Hub - $34.99', 'target_language': 'Spanish'}
  Result: Ratón inalámbrico - $29.99, Teclado mecánico - $79.99, Hub USB-C - $34.99

--- Step 4 ---
Final answer:
Here are the electronics products I found:

1. Wireless Mouse - $29.99
2. Mechanical Keyboard - $79.99
3. USB-C Hub - $34.99

The total cost of all products is **$144.97**.

Translated into Spanish, the product list is:

1. Ratón inalámbrico - $29.99
2. Teclado mecánico - $79.99
3. Hub USB-C - $34.99


'Here are the electronics products I found:\n\n1. Wireless Mouse - $29.99\n2. Mechanical Keyboard - $79.99\n3. USB-C Hub - $34.99\n\nThe total cost of all products is **$144.97**.\n\nTranslated into Spanish, the product list is:\n\n1. Ratón inalámbrico - $29.99\n2. Teclado mecánico - $79.99\n3. Hub USB-C - $34.99'

**What happened:** The model chained 3 tools: `search_products` (found products), `calculator` (totaled prices), and `translate_text` (translated the list). Each call depended on a previous result.

## 4. Nested Tool Use

**What it does:** A tool itself calls other tools inside it. One outer tool call triggers multiple inner tools behind the scenes.

**When to use it:** When you build complex tools from simpler ones, like a "travel planner" that internally uses weather and time tools.

**Steps:**
1. Create an outer tool that internally calls other tools.
2. Register the outer tool with the model.
3. The model calls the outer tool once.
4. The outer tool runs multiple inner tools behind the scenes.

In [16]:
def travel_advisor(city):
    print("  [travel_advisor] Getting info for:", city)

    # This tool internally uses other tools
    weather_result = get_weather(city)
    print("  [travel_advisor] Weather:", weather_result)

    
    timezone_map = {
        "new york": "EST",
        "london": "GMT",
        "tokyo": "JST",
        "paris": "CET",
        "sydney": "AEST",
    }

    city_lower = city.lower()
    time_result = "Time not available"

    if city_lower in timezone_map:
        timezone = timezone_map[city_lower]
        time_result = get_time(timezone)
        print("  [travel_advisor] Time:", time_result)

    
    # Use the model to create a travel summary
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Write a 2-sentence travel tip based on the weather and time info."
            },
            {
                "role": "user",
                "content": "City: " + city + "\nWeather: " + weather_result + "\nTime: " + time_result
            }
        ],
        max_tokens=100
    )
    
    tip = response.choices[0].message.content
    return "Weather: " + weather_result + "\nTime: " + time_result + "\nTip: " + tip

**What happened:** We created `travel_advisor`, which internally calls `get_weather` and `get_time`. This is nesting: one outer tool uses multiple inner tools and also calls the AI to generate a travel tip.

In [ ]:
# Register the nested tool
nested_tools = all_tools.copy()
nested_tools.append({
    "type": "function",
    "function": {
        "name": "travel_advisor",
        "description": "Get travel advice for a city including weather, time, and tips.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    }
})


def run_tool_with_nested(function_name, arguments):
    if function_name == "travel_advisor":
        return travel_advisor(arguments["city"])
    return run_tool(function_name, arguments)



# Test it
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Give me travel advice for Tokyo"}
    ],
    tools=nested_tools,
    tool_choice="auto"
)

message = response.choices[0].message

if message.tool_calls:
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    print("Model called:", function_name)
    print()
    
    result = run_tool_with_nested(function_name, arguments)
    
    print()
    print("Final result:")
    print(result)

Model called: travel_advisor

  [travel_advisor] Getting info for: Tokyo
  [travel_advisor] Weather: Tokyo: 68F, cloudy, 60% humidity
  [travel_advisor] Time: JST: 4:00 AM

Final result:
Weather: Tokyo: 68F, cloudy, 60% humidity
Time: JST: 4:00 AM
Tip: When visiting Tokyo at 4:00 AM with cloudy skies and 60% humidity, be sure to dress in layers to stay comfortable as temperatures can feel cooler with the moisture in the air. Also, consider bringing a light rain jacket just in case, as the weather can change throughout the day.


**What happened:** The model called `travel_advisor("Tokyo")`, which internally ran `get_weather` and `get_time`. One tool call triggered three tools behind the scenes.

## 5. Dynamic Tool Creation

**What it does:** The agent creates new tools at runtime by writing Python functions. Instead of a fixed set of tools, the agent invents tools on the fly.

**When to use it:** When you cannot predict what tools the agent will need ahead of time.

**Steps:**
1. The model writes a tool as a Python function.
2. We execute the code to define the function.
3. We test and use the new tool.

In [28]:
def ask_model_to_create_tool(task_description):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You create Python functions. "
                    "Write the function definition. No explanation. No markdown. "
                    "The function should take simple arguments and return a string. "
                    "Do not use any imports."
                )
            },
            {
                "role": "user",
                "content": "Create a function for: " + task_description
            }
        ],
        max_tokens=300
    )
    
    code_text = response.choices[0].message.content
    code_text = code_text.strip()
    print(code_text)


    # Remove markdown code fences if present
    if code_text.startswith("```"):
        lines = code_text.split("\n")
        lines = lines[1:-1]
        code_text = "\n".join(lines)

    return code_text


tool_code = ask_model_to_create_tool("Convert temperature from Celsius to Fahrenheit")
print("Generated tool code:")
print(tool_code)

def celsius_to_fahrenheit(celsius):
    fahrenheit = (celsius * 9/5) + 32
    return str(fahrenheit) + ' °F'
Generated tool code:
def celsius_to_fahrenheit(celsius):
    fahrenheit = (celsius * 9/5) + 32
    return str(fahrenheit) + ' °F'


**What happened:** The model wrote a temperature conversion function. It created a tool that did not exist before.

In [29]:
# Execute the generated code to define the function
exec(tool_code)
print()

# Find the function name from the generated code
function_name = ""
for line in tool_code.split("\n"):
    line = line.strip()
    if line.startswith("def "):
        function_name = line.split("(")[0].replace("def ", "")
        break

print("Created function:", function_name)

# Test the dynamically created function
dynamic_function = eval(function_name)
test_result = dynamic_function(100)
print("Test: 100C =", test_result)

test_result = dynamic_function(0)
print("Test: 0C =", test_result)

test_result = dynamic_function(37)
print("Test: 37C =", test_result)


Created function: celsius_to_fahrenheit
Test: 100C = 212.0 °F
Test: 0C = 32.0 °F
Test: 37C = 98.6 °F


**What happened:** We used `exec()` to define the generated function, then tested it with real inputs. Warning: running `exec()` on AI-generated code is dangerous in production -- always validate and sandbox.

In [26]:
tool_code_2 = ask_model_to_create_tool("Check if a password is strong (at least 8 chars, has uppercase, lowercase, and a number)")
print("Generated tool code:")
print(tool_code_2)
print()

exec(tool_code_2)

function_name_2 = ""
for line in tool_code_2.split("\n"):
    line = line.strip()
    if line.startswith("def "):
        function_name_2 = line.split("(")[0].replace("def ", "")
        break

dynamic_function_2 = eval(function_name_2)
print("Testing password checker:")
print("  'hello':", dynamic_function_2("hello"))
print("  'Hello123':", dynamic_function_2("Hello123"))
print("  'MyP@ssw0rd':", dynamic_function_2("MyP@ssw0rd"))

def is_strong_password(password):
    if len(password) < 8:
        return "Weak password"
    if not any(char.isupper() for char in password):
        return "Weak password"
    if not any(char.islower() for char in password):
        return "Weak password"
    if not any(char.isdigit() for char in password):
        return "Weak password"
    return "Strong password"
Generated tool code:
def is_strong_password(password):
    if len(password) < 8:
        return "Weak password"
    if not any(char.isupper() for char in password):
        return "Weak password"
    if not any(char.islower() for char in password):
        return "Weak password"
    if not any(char.isdigit() for char in password):
        return "Weak password"
    return "Strong password"

Testing password checker:
  'hello': Weak password
  'Hello123': Strong password
  'MyP@ssw0rd': Strong password


**What happened:** The model created a completely different tool -- a password strength checker. The agent can invent any tool it needs.

## 6. Tool Retrieval

**What it does:** When you have many tools (50+), you first find the most relevant ones for the current question using embeddings (numerical representations of meaning), then send only those few tools to the model.

**When to use it:** When your agent has many tools but each question only needs 2-3. This keeps context small and the model focused.

**Steps:**
1. Store embeddings for all tool descriptions.
2. When a question comes in, compare its embedding to tool embeddings.
3. Send only the top matching tools to the model.

In [30]:
# A large tool registry with descriptions
tool_registry = {
    "calculator": "Calculate math expressions, do arithmetic, solve equations",
    "get_weather": "Get weather forecast, temperature, humidity, conditions for a city",
    "get_time": "Get current time in different timezones around the world",
    "translate_text": "Translate text between languages, convert languages",
    "search_products": "Search for products, find items to buy, look up prices",
    "send_email": "Send an email message to someone, compose and deliver email",
    "calendar_schedule": "Schedule meetings, check calendar, book appointments",
    "file_manager": "Read, write, delete, list files on disk",
    "database_query": "Run SQL queries, search database, look up records",
    "image_generator": "Generate images from text descriptions",
    "code_executor": "Run Python code, execute scripts",
    "web_scraper": "Scrape web pages, extract content from URLs",
    "pdf_reader": "Read and extract text from PDF files",
    "sentiment_analyzer": "Analyze sentiment of text, detect emotions",
    "spell_checker": "Check spelling and grammar in text",
}

print("Tool registry has", len(tool_registry), "tools")
for name in tool_registry:
    print("  ", name, "-", tool_registry[name][:50])

Tool registry has 15 tools
   calculator - Calculate math expressions, do arithmetic, solve e
   get_weather - Get weather forecast, temperature, humidity, condi
   get_time - Get current time in different timezones around the
   translate_text - Translate text between languages, convert language
   search_products - Search for products, find items to buy, look up pr
   send_email - Send an email message to someone, compose and deli
   calendar_schedule - Schedule meetings, check calendar, book appointmen
   file_manager - Read, write, delete, list files on disk
   database_query - Run SQL queries, search database, look up records
   image_generator - Generate images from text descriptions
   code_executor - Run Python code, execute scripts
   web_scraper - Scrape web pages, extract content from URLs
   pdf_reader - Read and extract text from PDF files
   sentiment_analyzer - Analyze sentiment of text, detect emotions
   spell_checker - Check spelling and grammar in text


**What happened:** We created a registry of 15 tools. Sending all 15 every time would waste tokens and confuse the model.

In [31]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    result = []
    for item in response.data:
        result.append(item.embedding)
    return result


# Pre-compute embeddings for all tool descriptions
tool_names = list(tool_registry.keys())
tool_descriptions = list(tool_registry.values())
tool_embeddings = get_embeddings(tool_descriptions)

print("Computed embeddings for", len(tool_embeddings), "tools")
print("Each embedding has", len(tool_embeddings[0]), "dimensions")

Computed embeddings for 15 tools
Each embedding has 1536 dimensions


**What happened:** We computed an embedding (a list of numbers capturing meaning) for each tool's description. We use these to find tools that match a user's question.

In [ ]:
def cosine_similarity(vec_a, vec_b):
    dot_product = 0
    magnitude_a = 0
    magnitude_b = 0
    for i in range(len(vec_a)):
        dot_product = dot_product + vec_a[i] * vec_b[i]
        magnitude_a = magnitude_a + vec_a[i] * vec_a[i]
        magnitude_b = magnitude_b + vec_b[i] * vec_b[i]
    magnitude_a = magnitude_a ** 0.5
    magnitude_b = magnitude_b ** 0.5
    if magnitude_a == 0 or magnitude_b == 0:
        return 0
    similarity = dot_product / (magnitude_a * magnitude_b)
    return similarity



def retrieve_tools(question, top_k=3):
    question_embedding = get_embeddings([question])[0]

    scores = []
    for i in range(len(tool_names)):
        score = cosine_similarity(question_embedding, tool_embeddings[i])
        scores.append((tool_names[i], score))

    scores.sort(key=lambda x: x[1], reverse=True)

    top_tools = []
    for i in range(top_k):
        name = scores[i][0]
        score = scores[i][1]
        top_tools.append(name)
        print("  ", i + 1, ".", name, "(score:", round(score, 3), ")")

    
    return top_tools

**What happened:** We created `cosine_similarity` (measures how similar two embeddings are, 1.0 = identical) and `retrieve_tools` (finds the top 3 most relevant tools by comparing embeddings).

In [33]:
print("Question: What is 25 * 17?")
relevant = retrieve_tools("What is 25 * 17?")
print("Selected tools:", relevant)
print()

print("Question: Send a message to my boss about the meeting")
relevant = retrieve_tools("Send a message to my boss about the meeting")
print("Selected tools:", relevant)
print()

print("Question: What is the weather and time in Tokyo?")
relevant = retrieve_tools("What is the weather and time in Tokyo?")
print("Selected tools:", relevant)

Question: What is 25 * 17?
   1 . calculator (score: 0.244 )
   2 . image_generator (score: 0.151 )
   3 . database_query (score: 0.117 )
Selected tools: ['calculator', 'image_generator', 'database_query']

Question: Send a message to my boss about the meeting
   1 . send_email (score: 0.421 )
   2 . calendar_schedule (score: 0.42 )
   3 . spell_checker (score: 0.231 )
Selected tools: ['send_email', 'calendar_schedule', 'spell_checker']

Question: What is the weather and time in Tokyo?
   1 . get_time (score: 0.38 )
   2 . get_weather (score: 0.361 )
   3 . calendar_schedule (score: 0.15 )
Selected tools: ['get_time', 'get_weather', 'calendar_schedule']


**What happened:** The retrieval system found the right tools for each question: math -> `calculator`, email -> `send_email`, weather+time -> `get_weather` and `get_time`. Only 3 tools are sent instead of all 15.

## 7. MCP (Model Context Protocol)

**What it does:** MCP is a standard protocol (created by Anthropic) for connecting AI models to tools and data sources. All tools follow one universal format instead of custom code per tool.

**When to use it:** When you want tools that work with any AI system. MCP is supported by Claude, ChatGPT, and many others.

**Steps:**
1. **MCP Server** provides tools and data (like a database server).
2. **MCP Client** connects to servers and uses their tools (like Claude Code).
3. They communicate over stdio or HTTP using a standard format.

### Simulating an MCP Server

In [35]:
class MCPServer:
    def __init__(self, name, description):
        self.name = name
        self.description = description
        self.tools = {}
        self.resources = {}

    def register_tool(self, tool_name, tool_description, tool_function, parameters):
        self.tools[tool_name] = {
            "description": tool_description,
            "function": tool_function,
            "parameters": parameters
        }

    def register_resource(self, resource_name, resource_data):
        self.resources[resource_name] = resource_data

    def list_tools(self):
        result = []
        for name in self.tools:
            tool_info = {
                "name": name,
                "description": self.tools[name]["description"],
                "parameters": self.tools[name]["parameters"]
            }
            result.append(tool_info)
        return result

    def list_resources(self):
        result = []
        for name in self.resources:
            result.append(name)
        return result

    def call_tool(self, tool_name, arguments):
        if tool_name not in self.tools:
            return "Error: tool " + tool_name + " not found"
        tool_function = self.tools[tool_name]["function"]
        result = tool_function(**arguments)
        return result

    def read_resource(self, resource_name):
        if resource_name not in self.resources:
            return "Error: resource " + resource_name + " not found"
        return self.resources[resource_name]

print("MCPServer class defined")

MCPServer class defined


**What happened:** We created an MCPServer class with `register_tool`, `register_resource`, `list_tools`, `call_tool`, and `read_resource`. This is a simplified simulation -- real MCP servers communicate over stdio or HTTP.

### Create an MCP Server with tools and resources

In [36]:
# Create a weather MCP server
weather_server = MCPServer("weather-server", "Provides weather data and forecasts")

weather_server.register_tool(
    "get_current_weather",
    "Get current weather for a city",
    get_weather,
    {"city": {"type": "string", "description": "City name"}}
)

weather_server.register_resource("supported_cities", [
    "New York", "London", "Tokyo", "Paris", "Sydney"
])


# Create a math MCP server
math_server = MCPServer("math-server", "Provides calculation tools")

math_server.register_tool(
    "calculate",
    "Evaluate a math expression",
    calculator,
    {"expression": {"type": "string", "description": "Math expression"}}
)

print("Weather server tools:", weather_server.list_tools())
print()
print("Weather server resources:", weather_server.list_resources())
print()
print("Math server tools:", math_server.list_tools())

Weather server tools: [{'name': 'get_current_weather', 'description': 'Get current weather for a city', 'parameters': {'city': {'type': 'string', 'description': 'City name'}}}]

Weather server resources: ['supported_cities']

Math server tools: [{'name': 'calculate', 'description': 'Evaluate a math expression', 'parameters': {'expression': {'type': 'string', 'description': 'Math expression'}}}]


**What happened:** We created two MCP servers: `weather-server` (one tool + list of supported cities) and `math-server` (one calculator tool). Each server is independent.

### MCP Client: connect to servers and use their tools

In [37]:
class MCPClient:
    def __init__(self):
        self.servers = {}

    def connect(self, server):
        self.servers[server.name] = server
        print("Connected to:", server.name, "-", server.description)

    def get_all_tools(self):
        all_tool_list = []
        for server_name in self.servers:
            server = self.servers[server_name]
            tools = server.list_tools()
            for tool in tools:
                tool["server"] = server_name
                all_tool_list.append(tool)
        return all_tool_list

    def call_tool(self, server_name, tool_name, arguments):
        if server_name not in self.servers:
            return "Error: server " + server_name + " not connected"
        server = self.servers[server_name]
        result = server.call_tool(tool_name, arguments)
        return result


# Create client and connect to both servers
mcp_client = MCPClient()
mcp_client.connect(weather_server)
mcp_client.connect(math_server)

print()
print("All available tools across all servers:")
all_mcp_tools = mcp_client.get_all_tools()
for tool in all_mcp_tools:
    print("  [" + tool["server"] + "]", tool["name"], "-", tool["description"])

Connected to: weather-server - Provides weather data and forecasts
Connected to: math-server - Provides calculation tools

All available tools across all servers:
  [weather-server] get_current_weather - Get current weather for a city
  [math-server] calculate - Evaluate a math expression
